In [1]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import imblearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [2]:
test = pd.read_csv("UNSW_NB15_testing-set.csv")
test = test.iloc[:,:-1] 

In [3]:
train = pd.read_csv("UNSW_NB15_training-set.csv")
train = train.iloc[:,:-1] 

In [4]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [5]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [6]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [7]:
y = train.pop("attack_cat").values
X = train.values

In [8]:
train.dtypes

id                     int64
dur                  float64
proto                  int32
service                int32
state                  int32
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_l

In [9]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [10]:
!pip install mlxtend

In [11]:
from mlxtend.feature_selection import SequentialFeatureSelector as sfs


In [12]:
clf = RandomForestClassifier(n_estimators=100, n_jobs=-1)

sfs1 = sfs(clf,
           k_features=10,
           forward=True,
           floating=False,
           verbose=2,
           scoring='accuracy',
           cv=5)

sfs1 = sfs1.fit(X, y)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   13.4s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  43 out of  43 | elapsed:  3.4min finished

[2022-05-01 19:42:46] Features: 1/10 -- score: 0.7342457345669405[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.6s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  42 out of  42 | elapsed:  4.1min finished

[2022-05-01 19:46:54] Features: 2/10 -- score: 0.7816273588957302[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.9s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  41 out of  41 | elapsed:  4.3min finished

[2022-05-01 19:51:11] Features: 3/10 -- score: 0.795133534555096[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   

In [13]:
X_train_sfs, X_test_sfs, y_train_sfs, y_test_sfs = train_test_split(X[:, list(sfs1.k_feature_idx_)], y, test_size=0.2, random_state=42)

In [14]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train_sfs, y_train_sfs)


KNeighborsClassifier()

In [15]:
y_pred_sfs = neigh.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 7539, 5: 3648, 3: 2459, 4: 1128, 2: 791, 7: 650, 0: 135, 1: 68, 8: 47, 9: 2})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [16]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[  24    9   30   50   18    0    0    0    0    0]
 [  16    5   20   40   30    0    2    3    1    0]
 [  18    8  300  368   41    5   29   13    4    0]
 [  42   23  298 1634   74   10  125   56   12    1]
 [  34   18   50  119  796   17  157   21    0    0]
 [   0    1   18   56   28 3580   35    3    2    0]
 [   0    1   31   69  109   28 7136   41    3    0]
 [   1    3   43  108   29    4   40  493    2    0]
 [   0    0    1   10    3    3   15   20   23    0]
 [   0    0    0    5    0    1    0    0    0    1]]
Accuracy Score : 0.8496993987975952
Report : 
              precision    recall  f1-score   support

           0       0.18      0.18      0.18       131
           1       0.07      0.04      0.05       117
           2       0.38      0.38      0.38       786
           3       0.66      0.72      0.69      2275
           4       0.71      0.66      0.68      1212
           5       0.98      0.96      0.97      3723
           6       0.95  

In [17]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.8453655203826008
Report : 
              precision    recall  f1-score   support

           0       0.18      0.16      0.17       546
           1       0.08      0.07      0.07       466
           2       0.38      0.40      0.39      3303
           3       0.64      0.70      0.67      8857
           4       0.70      0.63      0.66      4850
           5       0.98      0.96      0.97     15148
           6       0.95      0.96      0.96     29582
           7       0.76      0.70      0.73      2773
           8       0.44      0.24      0.31       303
           9       0.27      0.11      0.15        37

    accuracy                           0.85     65865
   macro avg       0.54      0.49      0.51     65865
weighted avg       0.84      0.85      0.84     65865



In [18]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train_sfs, y_train_sfs)

SVC(decision_function_shape='ovo', gamma='auto')

In [19]:
y_pred_sfs=clf.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 9628, 5: 4492, 3: 1652, 4: 685, 2: 10})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [20]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[   0    0    0    3   13   73   42    0    0    0]
 [   0    0    0    4    7   85   21    0    0    0]
 [   0    0    8  122   26   91  539    0    0    0]
 [   0    0    1 1255   53  187  779    0    0    0]
 [   0    0    0   11  489  262  450    0    0    0]
 [   0    0    0   44   37 3439  203    0    0    0]
 [   0    0    1  131   35  319 6932    0    0    0]
 [   0    0    0   79   22   32  590    0    0    0]
 [   0    0    0    0    3    2   70    0    0    0]
 [   0    0    0    3    0    2    2    0    0    0]]
Accuracy Score : 0.7361996720714156
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.80      0.01      0.02       786
           3       0.76      0.55      0.64      2275
           4       0.71      0.40      0.52      1212
           5       0.77      0.92      0.84      3723
           6       0.72  

In [21]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.7340469141425643
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.70      0.01      0.02      3303
           3       0.73      0.54      0.62      8857
           4       0.72      0.41      0.52      4850
           5       0.76      0.92      0.84     15148
           6       0.72      0.93      0.81     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.36      0.28      0.28     65865
weighted avg       0.69      0.73      0.68     65865



In [22]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train_sfs, y_train_sfs)

In [23]:
y_pred_sfs = clf.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 11915, 5: 3920, 3: 449, 4: 181, 2: 2})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [24]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Accuracy Score : 0.6269508714398494
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       1.00      0.00      0.01       786
           3       0.52      0.10      0.17      2275
           4       0.50      0.08      0.13      1212
           5       0.75      0.79      0.77      3723
           6       0.59      0.95      0.73      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.63     16467
   macro avg       0.34      0.19      0.18     16467
weighted avg       0.59      0.63      0.54     16467



In [25]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.6318074850072117
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.60      0.00      0.00      3303
           3       0.54      0.10      0.17      8857
           4       0.57      0.09      0.15      4850
           5       0.75      0.80      0.78     15148
           6       0.60      0.95      0.73     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.63     65865
   macro avg       0.31      0.19      0.18     65865
weighted avg       0.58      0.63      0.54     65865



In [26]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train_sfs, y_train_sfs)
clf

MLPClassifier(alpha=1)

In [27]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[   0    0    0    0    7   69   55    0    0    0]
 [   0    0    0    1    7   76   33    0    0    0]
 [   0    0    2   38    8   81  657    0    0    0]
 [   0    0    0  235   15  178 1847    0    0    0]
 [   0    0    0   24   91  253  844    0    0    0]
 [   0    0    0   11   41 2947  724    0    0    0]
 [   0    0    0   62    5  302 7049    0    0    0]
 [   0    0    0   73    6   13  631    0    0    0]
 [   0    0    0    0    1    1   73    0    0    0]
 [   0    0    0    5    0    0    2    0    0    0]]
Accuracy Score : 0.6269508714398494
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       1.00      0.00      0.01       786
           3       0.52      0.10      0.17      2275
           4       0.50      0.08      0.13      1212
           5       0.75      0.79      0.77      3723
           6       0.59  

In [28]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.7072193122295605
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.45      0.00      0.00      3303
           3       0.72      0.49      0.59      8857
           4       0.65      0.21      0.31      4850
           5       0.75      0.90      0.82     15148
           6       0.69      0.93      0.79     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.71     65865
   macro avg       0.33      0.25      0.25     65865
weighted avg       0.65      0.71      0.65     65865



In [29]:
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train_sfs, y_train_sfs)
arr = forest.predict(X_train_sfs).astype(int)
print(classification_report(y_train_sfs, arr))
print('Accuracy Score :',accuracy_score(y_train_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.08      0.15       546
           1       0.67      0.06      0.12       466
           2       0.32      0.93      0.48      3303
           3       0.85      0.50      0.63      8857
           4       0.82      0.68      0.74      4850
           5       0.73      0.97      0.83     15148
           6       0.96      0.82      0.89     29582
           7       0.88      0.55      0.68      2773
           8       0.29      0.02      0.04       303
           9       0.00      0.00      0.00        37

    accuracy                           0.78     65865
   macro avg       0.65      0.46      0.45     65865
weighted avg       0.84      0.78      0.78     65865

Accuracy Score : 0.7791695133986184


In [30]:
arr = forest.predict(X_test_sfs).astype(int)
print(classification_report(y_test_sfs, arr))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.90      0.07      0.13       131
           1       0.12      0.01      0.02       117
           2       0.27      0.83      0.41       786
           3       0.71      0.37      0.49      2275
           4       0.63      0.57      0.60      1212
           5       0.68      0.97      0.80      3723
           6       0.96      0.78      0.86      7418
           7       0.93      0.53      0.68       723
           8       0.50      0.03      0.05        75
           9       0.00      0.00      0.00         7

    accuracy                           0.73     16467
   macro avg       0.57      0.42      0.40     16467
weighted avg       0.80      0.73      0.73     16467

Accuracy Score : 0.7293981903200341


In [31]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train_sfs, y_train_sfs)
arr = forest.predict(X_train_sfs).astype(int)
print(classification_report(y_train_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.08      0.15       546
           1       0.67      0.06      0.12       466
           2       0.32      0.93      0.48      3303
           3       0.85      0.50      0.63      8857
           4       0.82      0.68      0.74      4850
           5       0.73      0.97      0.83     15148
           6       0.96      0.82      0.89     29582
           7       0.88      0.55      0.68      2773
           8       0.29      0.02      0.04       303
           9       0.00      0.00      0.00        37

    accuracy                           0.78     65865
   macro avg       0.65      0.46      0.45     65865
weighted avg       0.84      0.78      0.78     65865



In [32]:
arr = classifier.predict(X_test_sfs).astype(int)
print(classification_report(y_test_sfs, arr))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.02      0.73      0.05       131
           1       0.00      0.00      0.00       117
           2       0.16      0.01      0.02       786
           3       0.33      0.16      0.22      2275
           4       0.24      0.05      0.09      1212
           5       0.87      0.53      0.66      3723
           6       0.81      0.43      0.56      7418
           7       0.01      0.01      0.01       723
           8       0.02      0.99      0.03        75
           9       0.01      0.43      0.01         7

    accuracy                           0.35     16467
   macro avg       0.25      0.33      0.16     16467
weighted avg       0.63      0.35      0.44     16467

Accuracy Score : 0.3489403048521285


In [33]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train_sfs, y_train_sfs)
y_pred = clf_entropy.predict(X_train_sfs)
print(classification_report(y_train_sfs, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.49      0.60      0.54      8857
           4       0.00      0.00      0.00      4850
           5       1.00      0.96      0.98     15148
           6       0.69      0.95      0.80     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.22      0.25      0.23     65865
weighted avg       0.61      0.73      0.66     65865



In [34]:
y_pred = clf_entropy.predict(X_test_sfs)
print(classification_report(y_test_sfs, y_pred))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.51      0.60      0.55      2275
           4       0.00      0.00      0.00      1212
           5       1.00      0.96      0.98      3723
           6       0.69      0.95      0.80      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.73     16467
   macro avg       0.22      0.25      0.23     16467
weighted avg       0.61      0.73      0.66     16467

Accuracy Score : 0.3489403048521285


In [35]:
#DecisionTree Gini Index
clf_gini = DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=3, min_samples_leaf=5)
clf_gini.fit(X_train_sfs, y_train_sfs)
y_pred = clf_gini.predict(X_train_sfs)
print(classification_report(y_train_sfs, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.83      0.43      0.56      8857
           4       0.00      0.00      0.00      4850
           5       1.00      0.96      0.98     15148
           6       0.63      1.00      0.77     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.25      0.24      0.23     65865
weighted avg       0.62      0.73      0.65     65865



In [36]:
y_pred = clf_gini.predict(X_test_sfs)
print(classification_report(y_test_sfs, y_pred))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.84      0.44      0.58      2275
           4       0.00      0.00      0.00      1212
           5       1.00      0.96      0.98      3723
           6       0.63      1.00      0.78      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.73     16467
   macro avg       0.25      0.24      0.23     16467
weighted avg       0.63      0.73      0.65     16467

Accuracy Score : 0.3489403048521285
